# Nome do Projeto

Aprendizado de Máquina

Prof. Dr. Leandro Augusto da Silva

---

Grupo:

## Preparação do ambiente

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import torch;
import numpy as np;
import pandas as pd;
import seaborn as sns;
import matplotlib as plt;
from transformers import set_seed;
from transformers import AutoTokenizer, AutoModel;

/home/joao/github/political-guidance-through-discourse/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Leitura do dataset

In [3]:
df = pd.read_json("content/firearm-carry-speeches-2014-2026.jsonl", lines=True)
df.shape

(256, 20)

## Análise Exploratória - TO DO

In [4]:
df.head()

,id,event_id,deputy_id,deputy_name,party,political_spectrum,ideological_stance,state,legislature_id,speech_type,keywords,transcript,summary,start_date,end_date,major_claim,micro_arguments,topic_consistency,is_argumentative,stance
0,a660973a-7e8f-422e-ac12-3d85e2ca27aa,35538,167614,Severino Ninho,PSB,Centro-esquerda,Progressista,PE,54,ORIENTAÇÃO DE BANCADA,"PL 6565/2013, PROJETO DE LEI, CONCESSÃO, PORTE...",O SR. SEVERINO NINHO (PSB-PE. Pela ordem. Sem ...,Orientação de bancada favorável do PSB ao Proj...,2014-03-26T14:00,2014-03-26T20:28,o PSB também vota a favor do projeto,[{'claim': 'alertando nossos queridos guardas ...,TOPIC,True,PRO
1,9bf5cbca-b734-421d-852e-54d48b21079d,35538,4931,Izalci,PSDB,Centro,Neutro,DF,54,ORIENTAÇÃO DE BANCADA,"PL 6565/2013, PROJETO DE LEI, CONCESSÃO, PORTE...",O SR. IZALCI (PSDB-DF. Pela ordem. Sem revisão...,Orientação de bancada favorável do PSDB ao Pro...,2014-03-26T14:00,2014-03-26T20:28,o PSDB vota pela aprovação do projeto,[{'claim': 'Os agentes e guardas prisionais pr...,TOPIC,True,PRO
2,1152aead-d96e-475b-8b09-bcbf9010d313,35538,73466,Rubens Bueno,PPS,,,PR,54,PELA ORDEM,"PL 6565/2013, PROJETO DE LEI, CONCESSÃO, PORTE...",O SR. RUBENS BUENO (PPS-PR. Pela Ordem. Sem re...,"Ponderações sobre o Projeto de Lei nº 6.565, d...",2014-03-26T14:00,2014-03-26T20:28,O PPS vota favoravelmente.,[{'claim': 'Nós também estamos de acordo com q...,TOPIC,True,PRO
3,4962cc8d-5fad-43a4-abdf-1c443171f23e,35538,73433,Arlindo Chinaglia,PT,Esquerda,Progressista,SP,54,PELA ORDEM,"PL 6565/2013, PROJETO DE LEI, CONCESSÃO, PORTE...",O SR. ARLINDO CHINAGLIA (PT-SP. Pela ordem. Se...,Solicitação à bancada do PT de retirada de req...,2014-03-26T14:00,2014-03-26T20:28,nós caminharemos para a votação de acordo com ...,"[{'claim': 'eu quero, então, propor e pedir à ...",TOPIC,True,NEUTRAL
4,1a79dbf5-a338-4deb-8ee2-646cc7b19b87,35538,74848,Jandira Feghali,PCDOB,Esquerda,Progressista,RJ,54,ORIENTAÇÃO DE BANCADA,"PL 6565/2013, PROJETO DE LEI, CONCESSÃO, PORTE...",A SRA. JANDIRA FEGHALI (PCdoB-RJ. Pela ordem. ...,Orientação de bancada favorável do PC do B ao ...,2014-03-26T14:00,2014-03-26T20:28,"O PCdoB vota ""sim"" ao projeto.",[{'claim': 'entendemos que é correto colocar n...,TOPIC,True,PRO


In [5]:
df.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 256 entries, 0 to 255
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  256 non-null    str   
 1   event_id            256 non-null    int64 
 2   deputy_id           256 non-null    int64 
 3   deputy_name         256 non-null    str   
 4   party               256 non-null    str   
 5   political_spectrum  256 non-null    str   
 6   ideological_stance  256 non-null    str   
 7   state               256 non-null    str   
 8   legislature_id      256 non-null    int64 
 9   speech_type         256 non-null    str   
 10  keywords            256 non-null    str   
 11  transcript          256 non-null    str   
 12  summary             256 non-null    str   
 13  start_date          256 non-null    str   
 14  end_date            256 non-null    str   
 15  major_claim         256 non-null    str   
 16  micro_arguments     256 non-null    o

## Transformação dos dados

### Obtenção do argumento principal em forma de texto corrido

In [6]:
def get_argument(adu: dict) -> str:
    plain_text = adu["major_claim"].capitalize();

    for micro_argument in adu["micro_arguments"]:
        plain_text += " " + micro_argument["claim"].capitalize();

        for premise in micro_argument["premises"]:
            if premise.upper() == "[IMPLICIT]":
                continue;
            
            plain_text += " " + premise.capitalize();

    return plain_text;

In [7]:
df["argument"] = df.apply(get_argument, axis=1);
df["argument"][0]

'O psb também vota a favor do projeto Alertando nossos queridos guardas e agentes penitenciários para a responsabilidade que se deve ter com uma arma na mão e também dentro de casa Pelo risco aos familiares É preciso que quem recebe o porte de arma receba um treinamento sério O psb se preocupa muito com essa questão de termos cada vez mais armas nas ruas É necessário que os próprios agentes de segurança prisional busquem esse treinamento Para que a arma não se volte contra eles e não prejudique a sociedade'

In [8]:
df.shape

(256, 21)

## Geração de embeddings via BERTimbau 

### Inicialização do tokenizer e do modelo

In [4]:
set_seed(34)

In [5]:
device = "cpu" # torch.device("cuda" if torch.cuda.is_available() else "cpu");
print(f"Hardware utilizado: {device}");

Hardware utilizado: cpu


In [6]:
model = "neuralmind/bert-large-portuguese-cased";
tokenizer = AutoTokenizer.from_pretrained(model);
bertimbau = AutoModel.from_pretrained(model).to(device);
max_length = bertimbau.config.max_position_embeddings;

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 15261.38it/s]
[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Tokenização

In [7]:
def tokenize(texts: list[str], max_length, device):
    return tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=max_length).to(device);

In [13]:
arguments = df["argument"].tolist();

In [14]:
argument_tokens = tokenize(arguments, max_length, device);

### Geração de embeddings

In [8]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state;
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float();

    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9);

In [ ]:
with torch.no_grad():
    argument_output = bertimbau(**argument_tokens);
    argument_embeddings = mean_pooling(argument_output, argument_tokens["attention_mask"]);

In [17]:
argument_array = argument_embeddings.cpu().numpy()

In [ ]:
embedding_artifact = {
    "embeddings": argument_embeddings.cpu(),
    "ids": df["id"].tolist(),
    "deputy_ids": df["deputy_id"].tolist(),
    "arguments": df["argument"].tolist(),
    "model_name": model,
    "max_length": max_length,
}

torch.save(embedding_artifact, "argument_embeddings.pt")
print(f"Embeddings salvos: {argument_embeddings.shape}")

## Embeddings das notícias

As notícias usam o mesmo BERTimbau, tokenização, `mean pooling` e configuração de dispositivo empregados nos argumentos. O texto de entrada combina o título e o corpo da notícia.

In [3]:
news_df = pd.read_json("content/noticia_partidos.jsonl", lines=True)

news_df["titulo"] = news_df["titulo"].fillna("").astype(str)
news_df["texto_cru"] = news_df["texto_cru"].fillna("").astype(str)
news_df["partido"] = news_df["partido"].fillna("").astype(str)
news_df["data_publicacao"] = news_df["data_publicacao"].fillna("").astype(str)
news_df["link"] = news_df["link"].fillna("").astype(str)

news_df["news_text"] = (
    news_df["titulo"].str.strip() + ". " + news_df["texto_cru"].str.strip()
).str.strip()

print(f"Notícias carregadas: {len(news_df)}")
news_df[["titulo", "partido", "news_text"]].head(2)

Notícias carregadas: 1061


,titulo,partido,news_text
0,Câmara dos Deputados aprova aumento da pena de...,AVANTE,Câmara dos Deputados aprova aumento da pena de...
1,Manaus reforça liderança nacional em segurança...,AVANTE,Manaus reforça liderança nacional em segurança...


In [9]:
news_texts = news_df["news_text"].tolist()
news_embeddings_batches = []
batch_size = 8

with torch.no_grad():
    for start in range(0, len(news_texts), batch_size):
        batch_texts = news_texts[start:start + batch_size]
        batch_tokens = tokenize(batch_texts, max_length, device)
        batch_output = bertimbau(**batch_tokens)
        batch_embeddings = mean_pooling(
            batch_output,
            batch_tokens["attention_mask"]
        )
        news_embeddings_batches.append(batch_embeddings.cpu())

news_embeddings = torch.cat(news_embeddings_batches, dim=0)
print(f"Embeddings das notícias: {tuple(news_embeddings.shape)}")

Embeddings das notícias: (1061, 1024)


In [ ]:
news_embedding_artifact = {
    "embeddings": news_embeddings,
    "titles": news_df["titulo"].tolist(),
    "texts": news_df["texto_cru"].tolist(),
    "news_texts": news_df["news_text"].tolist(),
    "parties": news_df["partido"].tolist(),
    "publication_dates": news_df["data_publicacao"].tolist(),
    "links": news_df["link"].tolist(),
    "model_name": model,
    "max_length": max_length,
    "pooling": "mean_pooling",
}

torch.save(news_embedding_artifact, "news_embeddings.pt")
print("Embeddings salvos em news_embeddings.pt")

Embeddings salvos em news_embeddings.pt
